In [2]:
import os
import zipfile
import numpy as np
import pandas as pd
import librosa
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report

# Unzip dataset if not already done
dataset_zip = '/content/drive/MyDrive/RAVDESS(dataset).zip'
extract_path = '/content/RAVDESS'
if not os.path.exists(extract_path):
    zipfile.ZipFile(dataset_zip, 'r').extractall(extract_path)

# Extract features (MFCC, pitch, energy)
def extract_features(file):
    y, sr = librosa.load(file, sr=None)
    mfcc = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13).T, axis=0)
    pitch = np.mean(librosa.piptrack(y=y, sr=sr)[0])
    energy = np.mean(librosa.feature.rms(y=y).T)
    return [pitch, energy] + mfcc.tolist()

# Load and process data
emotions_map = {'03': 'happy', '05': 'angry', '01': 'sad'}
features, labels, files = [], [], []

for root, _, audio_files in os.walk(extract_path):
    for file in audio_files:
        if file.endswith(".wav"):
            emotion = emotions_map.get(file.split('-')[2])
            if emotion:
                features.append(extract_features(os.path.join(root, file)))
                labels.append(emotion)
                files.append(file)

# Save features to CSV
columns = ['File_Name'] + ['Pitch', 'Energy'] + [f'MFCC_{i+1}' for i in range(13)] + ['Emotion']
df = pd.DataFrame(features, columns=columns[1:-1])
df.insert(0, 'File_Name', files)
df['Emotion'] = labels
df.to_csv('/content/features_with_emotions.csv', index=False)
print("Features saved to features_with_emotions.CSV")

# Train and evaluate model
X_train, X_test, y_train, y_test = train_test_split(
    features, LabelEncoder().fit_transform(labels), test_size=0.2, random_state=42, stratify=labels)

scaler = StandardScaler().fit(X_train)
model = SVC(kernel='linear', random_state=42).fit(scaler.transform(X_train), y_train)

# Evaluate model
y_pred = model.predict(scaler.transform(X_test))
print(classification_report(y_test, y_pred, target_names=['happy', 'angry', 'sad']))


Features saved to features_with_emotions.CSV
              precision    recall  f1-score   support

       happy       0.81      0.75      0.78        77
       angry       0.76      0.73      0.74        77
         sad       0.74      0.89      0.81        38

    accuracy                           0.77       192
   macro avg       0.77      0.79      0.78       192
weighted avg       0.77      0.77      0.77       192

